In [2]:
%pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

def calcular_drawdown(retornos):
    """Calcula o Maximum Drawdown de uma série de retornos diários."""
    curva_acumulada = (1 + retornos).cumprod()
    picos = curva_acumulada.cummax()
    drawdowns = (curva_acumulada - picos) / picos
    return drawdowns.min()

def calcular_sortino(retornos, taxa_livre_risco_diaria, cagr, cagr_rf, fator_anualizacao=252):
    """Calcula o Índice de Sortino Anualizado."""
    # Retornos abaixo da taxa livre de risco (Downside)
    downside = retornos - taxa_livre_risco_diaria
    downside = downside[downside < 0]
    
    # Se não houver downside (raro), o Sortino tende ao infinito
    if len(downside) == 0:
        return np.inf
        
    # Desvio Padrão do Downside (Downside Deviation) anualizado
    downside_deviation = np.sqrt(np.mean(downside**2)) * np.sqrt(fator_anualizacao)
    
    # Sortino = (Retorno da Carteira - Retorno Risk Free) / Downside Deviation
    sortino = (cagr - cagr_rf) / downside_deviation
    return sortino

def gerar_tabela_metricas_finais(diretorio_dados):
    print("=== GERAÇÃO DA TABELA DE MÉTRICAS INSTITUCIONAIS ===")
    
    # 1. Carregamento e Preparação
    historico_pesos = pd.read_csv(os.path.join(diretorio_dados, "historico_alocacao_lstm_ff5_overnight.csv"), index_col=0, parse_dates=True)
    df_retornos = pd.read_csv(os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv"), index_col='Data', parse_dates=True)
    historico_turnover = pd.read_csv(os.path.join(diretorio_dados, "historico_turnover_lstm_ff5_overnight.csv"), index_col=0, header=None, parse_dates=True)
    historico_turnover.columns = ['Turnover']
    
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    data_inicio = historico_pesos.index[0]
    df_retornos = df_retornos.loc[data_inicio:]
    
    pesos_diarios = historico_pesos.reindex(df_retornos.index).ffill().bfill()
    colunas_ativos = [col for col in historico_pesos.columns if col in df_retornos.columns]
    
    # 2. Cálculo dos Retornos Diários Brutos
    retorno_carteira_bruto = (df_retornos[colunas_ativos].values * pesos_diarios[colunas_ativos].values).sum(axis=1)
    retorno_carteira_bruto = pd.Series(retorno_carteira_bruto, index=df_retornos.index)
    
    # 3. Correção de Calendário e Custos (A mesma blindagem de sempre)
    custo_transacional = 0.002
    custos_diarios = pd.Series(0.0, index=retorno_carteira_bruto.index)
    historico_turnover = historico_turnover[historico_turnover.index.notna()]
    
    datas_validas = []
    for data in historico_turnover.index:
        if data in custos_diarios.index:
            datas_validas.append(data)
        else:
            dias_anteriores = custos_diarios.index[custos_diarios.index <= data]
            if not dias_anteriores.empty:
                datas_validas.append(dias_anteriores.max())
            else:
                dias_posteriores = custos_diarios.index[custos_diarios.index >= data]
                if not dias_posteriores.empty:
                    datas_validas.append(dias_posteriores.min())
                else:
                    datas_validas.append(pd.NaT)
                    
    historico_turnover.index = datas_validas
    historico_turnover = historico_turnover[historico_turnover.index.notna()]
    custos_diarios.loc[historico_turnover.index] = historico_turnover['Turnover'] * custo_transacional
    
    # Retornos Finais Líquidos
    retorno_carteira_liquido = retorno_carteira_bruto - custos_diarios
    retorno_ibov = df_retornos['IBOV']
    retorno_cdi = df_cdi['CDI'].reindex(df_retornos.index).ffill()
    
    # =========================================================================
    # 4. CÁLCULO DAS 4 GRANDES MÉTRICAS (ANUALIZADAS)
    # =========================================================================
    dias_totais = len(retorno_carteira_liquido)
    fator_anualizacao = 252
    
    # A. Retornos (Acumulado e CAGR)
    ret_acum_est = (1 + retorno_carteira_liquido).cumprod().iloc[-1] - 1
    ret_acum_ibov = (1 + retorno_ibov).cumprod().iloc[-1] - 1
    ret_acum_cdi = (1 + retorno_cdi).cumprod().iloc[-1] - 1
    
    cagr_est = (1 + ret_acum_est) ** (fator_anualizacao / dias_totais) - 1
    cagr_ibov = (1 + ret_acum_ibov) ** (fator_anualizacao / dias_totais) - 1
    cagr_cdi = (1 + ret_acum_cdi) ** (fator_anualizacao / dias_totais) - 1
    
    # B. Volatilidade Anualizada (Risco Total / Variância)
    vol_est = retorno_carteira_liquido.std() * np.sqrt(fator_anualizacao)
    vol_ibov = retorno_ibov.std() * np.sqrt(fator_anualizacao)
    
    # C. Maximum Drawdown (Risco de Cauda / Queda Máxima)
    mdd_est = calcular_drawdown(retorno_carteira_liquido)
    mdd_ibov = calcular_drawdown(retorno_ibov)
    
    # D. Índice Sharpe (Prêmio por Risco Total)
    sharpe_est = (cagr_est - cagr_cdi) / vol_est if vol_est > 0 else 0
    sharpe_ibov = (cagr_ibov - cagr_cdi) / vol_ibov if vol_ibov > 0 else 0
    
    # E. Índice Sortino (Prêmio por Risco de Baixa / Downside)
    sortino_est = calcular_sortino(retorno_carteira_liquido, retorno_cdi, cagr_est, cagr_cdi)
    sortino_ibov = calcular_sortino(retorno_ibov, retorno_cdi, cagr_ibov, cagr_cdi)
    
    # Montar o DataFrame Final para o TCC
    df_metricas = pd.DataFrame({
        'Métrica': [
            '1. Retorno Acumulado', 
            '2. Retorno Anual (CAGR)', 
            '3. Volatilidade Anualizada', 
            '4. Maximum Drawdown', 
            '5. Índice de Sharpe',
            '6. Índice de Sortino'
        ],
        'Estratégia Híbrida (LSTM+FF5)': [
            f"{ret_acum_est*100:.2f}%", 
            f"{cagr_est*100:.2f}%", 
            f"{vol_est*100:.2f}%", 
            f"{mdd_est*100:.2f}%", 
            f"{sharpe_est:.2f}",
            f"{sortino_est:.2f}"
        ],
        'Benchmark (IBOVESPA)': [
            f"{ret_acum_ibov*100:.2f}%", 
            f"{cagr_ibov*100:.2f}%", 
            f"{vol_ibov*100:.2f}%", 
            f"{mdd_ibov*100:.2f}%", 
            f"{sharpe_ibov:.2f}",
            f"{sortino_ibov:.2f}"
        ]
    })
    
    df_metricas.set_index('Métrica', inplace=True)
    
    print("\n" + df_metricas.to_markdown())
    
    caminho_tabela = os.path.join(diretorio_dados, "tabela_metricas_finais_tcc.csv")
    df_metricas.to_csv(caminho_tabela, sep=';', encoding='utf-8-sig')
    
    print(f"\n✅ Tabela exportada com sucesso para: {caminho_tabela}")
    
    return df_metricas

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
tabela_final = gerar_tabela_metricas_finais(pasta_dados)

=== GERAÇÃO DA TABELA DE MÉTRICAS INSTITUCIONAIS ===

| Métrica                    | Estratégia Híbrida (LSTM+FF5)   | Benchmark (IBOVESPA)   |
|:---------------------------|:--------------------------------|:-----------------------|
| 1. Retorno Acumulado       | 56.48%                          | 243.49%                |
| 2. Retorno Anual (CAGR)    | 4.26%                           | 12.17%                 |
| 3. Volatilidade Anualizada | 38.56%                          | 23.31%                 |
| 4. Maximum Drawdown        | -92.99%                         | -46.82%                |
| 5. Índice de Sharpe        | -0.14                           | 0.10                   |
| 6. Índice de Sortino       | -0.15                           | 0.10                   |

✅ Tabela exportada com sucesso para: C:\VSCodeWorkspace\TCC_Escrito\data\tabela_metricas_finais_tcc.csv
